# 1. Imports

In [2]:
import os
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from pathlib import Path

# 2. Config

In [38]:
CONFIG = {
    "BASE_ROOT" : Path(r"C:\Users\tahmi\Documents\Work\Text2Sign\t2slt\tests\sign2text\BiLSTM_T\wlasl1000_2"),
    "SAVE_PATH" : Path(r"C:\Users\tahmi\Documents\Work\Text2Sign\t2slt\tests\sign2text\BiLSTM_T\wlasl1000_2\index.pkl"),
    
    "ROOT": Path(r"E:\WLASL\wlasl_1000_preproc"),
    "VIDEO_ROOT": Path(r"E:\WLASL\wlasl_1000_preproc\videos"),

    "TRAIN_JSON": Path(r"E:\WLASL\wlasl_1000_preproc\train_final.json"),
    "VAL_JSON": Path(r"E:\WLASL\wlasl_1000_preproc\val_final.json"),
    "TEST_JSON": Path(r"E:\WLASL\wlasl_1000_preproc\test_final.json"),

    "LABEL_MAP_JSON": Path(r"E:\WLASL\wlasl_1000_preproc\label_map_final.json"),

    "INDEX_PATH": Path(r"E:\WLASL\wlasl_1000_preproc\video_index.pkl"),

    "MAX_LEN": 64,
    "BATCH_SIZE": 16
}

In [39]:
for name, value in CONFIG.items():
    if isinstance(value, Path):
        if value.exists():
            print(f"{name} exists at {value}")
        else:
            print(f"{name} does NOT exist at {value}")
    else:
        print(f"{name} = {value}")


BASE_ROOT exists at C:\Users\tahmi\Documents\Work\Text2Sign\t2slt\tests\sign2text\BiLSTM_T\wlasl1000_2
SAVE_PATH exists at C:\Users\tahmi\Documents\Work\Text2Sign\t2slt\tests\sign2text\BiLSTM_T\wlasl1000_2\index.pkl
ROOT exists at E:\WLASL\wlasl_1000_preproc
VIDEO_ROOT exists at E:\WLASL\wlasl_1000_preproc\videos
TRAIN_JSON exists at E:\WLASL\wlasl_1000_preproc\train_final.json
VAL_JSON exists at E:\WLASL\wlasl_1000_preproc\val_final.json
TEST_JSON exists at E:\WLASL\wlasl_1000_preproc\test_final.json
LABEL_MAP_JSON exists at E:\WLASL\wlasl_1000_preproc\label_map_final.json
INDEX_PATH exists at E:\WLASL\wlasl_1000_preproc\video_index.pkl
MAX_LEN = 64
BATCH_SIZE = 16


# 3. Load Label Map

In [27]:
import json

with open(CONFIG["LABEL_MAP_JSON"], "r") as f:
    label_map = json.load(f)

# 0-based indexing
gloss_to_idx = {v: int(k) - 1 for k, v in label_map.items()}
idx_to_gloss = {int(k) - 1: v for k, v in label_map.items()}

NUM_CLASSES = len(gloss_to_idx)

print("Num classes:", NUM_CLASSES)

Num classes: 1000


# 4. JSON Parser

vid_id --> Label

In [28]:
def parse_split(json_path, gloss_to_idx):
    import json

    with open(json_path, "r") as f:
        data = json.load(f)

    samples = []

    for entry in data:
        gloss = entry["gloss"]

        if gloss not in gloss_to_idx:
            continue

        label = gloss_to_idx[gloss]

        for inst in entry["instances"]:
            vid = inst["video_id"]
            samples.append((vid, label))

    return samples

### Build vidId -> Label Map

In [31]:
import os
import pickle
from tqdm import tqdm

def build_index(video_root, save_path):
    if save_path.exists():
        print("Loading cached index...")
        with open(save_path, "rb") as f:
            return pickle.load(f)

    video_index = {}

    print("Building video index...")

    for root, _, files in os.walk(video_root):
        for file in files:
            if file.endswith(".npy"):
                vid_id = file.replace(".npy", "")
                full_path = os.path.join(root, file)
                video_index[vid_id] = full_path

    with open(save_path, "wb") as f:
        pickle.dump(video_index, f)

    print("Index built:", len(video_index))
    return video_index


video_index = build_index(CONFIG["VIDEO_ROOT"], CONFIG["SAVE_PATH"])
print("Indexed videos:", len(video_index))

Building video index...
Index built: 6073
Indexed videos: 6073


# 5. Dataset

In [32]:
import numpy as np
import torch
from torch.utils.data import Dataset

class SignDataset(Dataset):
    def __init__(self, samples, video_index, max_len=64):
        self.samples = samples
        self.video_index = video_index
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vid, label = self.samples[idx]

        if vid not in self.video_index:
            # fallback => skip
            return self.__getitem__((idx + 1) % len(self))

        path = self.video_index[vid]

        try:
            data = np.load(path)  # (T, features)
        except:
            return self.__getitem__((idx + 1) % len(self))

        # Padding / Truncation
        T = data.shape[0]

        if T > self.max_len:
            data = data[:self.max_len]
        else:
            pad = np.zeros((self.max_len - T, data.shape[1]))
            data = np.concatenate([data, pad], axis=0)

        return torch.tensor(data, dtype=torch.float32), torch.tensor(label)

# 6. Dataloaders

In [33]:
train_samples = parse_split(CONFIG["TRAIN_JSON"], gloss_to_idx)
val_samples = parse_split(CONFIG["VAL_JSON"], gloss_to_idx)
test_samples = parse_split(CONFIG["TEST_JSON"], gloss_to_idx)

print("Train:", len(train_samples))
print("Val:", len(val_samples))
print("Test:", len(test_samples))

Train: 7191
Val: 1970
Test: 1457


In [34]:
video_index = build_index(CONFIG["VIDEO_ROOT"], CONFIG["INDEX_PATH"])

Building video index...
Index built: 6073


In [35]:
from torch.utils.data import DataLoader

train_dataset = SignDataset(train_samples, video_index, CONFIG["MAX_LEN"])
val_dataset = SignDataset(val_samples, video_index, CONFIG["MAX_LEN"])
test_dataset = SignDataset(test_samples, video_index, CONFIG["MAX_LEN"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False)

# 7. Utils

In [41]:
SAVE_PATH = CONFIG["BASE_ROOT"]

PLOTS_PATH = SAVE_PATH / "plots"
METRICS_PATH = SAVE_PATH / "metrics"
CONF_PATH = SAVE_PATH / "confusion"

for p in [SAVE_PATH, PLOTS_PATH, METRICS_PATH, CONF_PATH]:
    p.mkdir(parents=True, exist_ok=True)

### metrics functions

In [42]:
import csv

METRICS_FILE = METRICS_PATH / "metrics.csv"

def init_metrics_file():
    with open(METRICS_FILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "train_loss", "val_loss",
            "train_acc", "val_acc"
        ])

def log_metrics(epoch, train_loss, val_loss, train_acc, val_acc):
    with open(METRICS_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch,
            train_loss, val_loss,
            train_acc, val_acc
        ])

### lost + acc plots

In [43]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_metrics():
    df = pd.read_csv(METRICS_FILE)

    # Loss Plot
    plt.figure()
    plt.plot(df["epoch"], df["train_loss"], label="Train Loss")
    plt.plot(df["epoch"], df["val_loss"], label="Val Loss")
    plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curve")
    plt.savefig(PLOTS_PATH / "loss.png")
    plt.close()

    # Accuracy Plot
    plt.figure()
    plt.plot(df["epoch"], df["train_acc"], label="Train Acc")
    plt.plot(df["epoch"], df["val_acc"], label="Val Acc")
    plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Curve")
    plt.savefig(PLOTS_PATH / "accuracy.png")
    plt.close()

### confusion metrics

In [44]:
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

def save_confusion_matrix(y_true, y_pred, epoch):
    cm = confusion_matrix(y_true, y_pred)

    np.save(CONF_PATH / f"cm_epoch_{epoch}.npy", cm)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest')
    plt.title(f"Confusion Matrix Epoch {epoch}")
    plt.colorbar()

    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.savefig(CONF_PATH / f"cm_epoch_{epoch}.png")
    plt.close()

### normalized heatmap

In [45]:
def save_confusion_heatmap(y_true, y_pred, epoch):
    cm = confusion_matrix(y_true, y_pred)

    # Normalize
    cm = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-6)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest')
    plt.title(f"Normalized Confusion Matrix Epoch {epoch}")
    plt.colorbar()

    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.savefig(CONF_PATH / f"cm_norm_epoch_{epoch}.png")
    plt.close()

### precision recall f1

In [46]:
from sklearn.metrics import classification_report

def save_classification_report(y_true, y_pred, epoch):
    report = classification_report(y_true, y_pred, output_dict=True)

    import json
    with open(CONF_PATH / f"report_epoch_{epoch}.json", "w") as f:
        json.dump(report, f, indent=4)

### evaluation

In [47]:
def evaluate_model(model, loader, device):
    model.eval()

    y_true = []
    y_pred = []

    correct = 0
    total = 0
    loss_total = 0

    import torch.nn.functional as F

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss = F.cross_entropy(logits, y)

            preds = torch.argmax(logits, dim=1)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

            correct += (preds == y).sum().item()
            total += y.size(0)
            loss_total += loss.item()

    acc = correct / total
    loss_avg = loss_total / len(loader)

    return acc, loss_avg, y_true, y_pred

# 8. Model

In [48]:
import torch
import torch.nn as nn
import math

class BiLSTMTransformerModel(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        hidden_dim=384,
        nhead=8,
        num_lstm_layers=2,
        num_transformer_layers=2,
        max_len=512,
        dropout=0.1
    ):
        super().__init__()

        self.hidden_dim = hidden_dim

        # 1 - Frame Encoder
        self.frame_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # 2 - Positional Encoding 
        self.register_buffer("pos_encoding", self._build_pos_encoding(max_len, hidden_dim))

        # 3 - BiLSTM
        self.lstm = nn.LSTM(
            hidden_dim,
            hidden_dim // 2,
            num_layers=num_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_lstm_layers > 1 else 0
        )

        # 4 - Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True  
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_transformer_layers
        )

        # 5 - Attention Pooling 
        self.attn_pool = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # 6 - Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

        self._init_weights()

    # Positional Encoding
    # ---------------------------
    def _build_pos_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        return pe.unsqueeze(0)  # (1, T, D)

    # Masked Attention Pooling
    # -------------------------
    def attention_pool(self, x, mask=None):
        # x: (B, T, H)

        scores = self.attn_pool(x)  # (B, T, 1)

        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(-1), -1e9)

        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(weights * x, dim=1)

        return pooled

    # Forward
    # -----------------
    def forward(self, x, mask=None):
        """
        x: (B, T, D)
        mask: (B, T) → True = padded
        """

        B, T, _ = x.shape

        x = self.frame_encoder(x)

        x = x + self.pos_encoding[:, :T, :]

        x, _ = self.lstm(x)

        x = self.transformer(x, src_key_padding_mask=mask)

        x = self.attention_pool(x, mask)

        return self.classifier(x)

    # Weight Init
    # ----------------
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

In [49]:
def sanity_check_overfit(model_class, model_args, train_loader, device, max_epochs=200):
    print(f"\n{'='*60}\n == SANITY CHECK: Attempting to overfit a single batch\n{'='*60}")
    model = model_class(**model_args).to(device)

    single_batch = None
    for seqs, lbls in train_loader:
        if seqs is None:
            continue
        single_batch = (seqs, lbls)
        break

    if single_batch is None:
        print("Could not load a valid batch!"); return False
    
    seqs, lbls = single_batch[0].to(device), single_batch[1].to(device)
    print(f"   Batch size: {seqs.shape[0]}, Unique labels: {len(torch.unique(lbls))}")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(seqs)
        loss = criterion(outputs, lbls)
        if torch.isnan(loss).any().item():
            print("NaN loss detected; skipping step.")
            continue
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 20 == 0:
            acc = (outputs.argmax(1) == lbls).float().mean().item() * 100
            print(f"   Epoch {epoch+1:3d}: Loss={loss.item():.4f}, Acc={acc:.2f}%")
            if acc > 95:
                print(f"\n == > Overfitted in {epoch+1} epochs. Model can learn.")
                return True
    
    print(f"\n  ! == > Could not overfit. There is a fundamental issue.")
    return False

# 9. Train

In [ ]:
def run_training_pipeline(
    model_class,
    model_args,
    train_loader,
    val_loader,
    test_loader,
    num_classes,
    save_dir,
    device="cuda",
    epochs=50
):
    import os
    import torch
    import json

    os.makedirs(save_dir, exist_ok=True)

    BEST_MODEL_PATH = os.path.join(save_dir, "best_model.pth")
    FINAL_RESULTS_PATH = os.path.join(save_dir, "final_test_results.json")

    # 1- Sanity Check
    print("\n == Running sanity check...")
    if not sanity_check_overfit(model_class, model_args, train_loader, device):
        print(" == Sanity check failed. Fix model/data first.")
        return

    print(" == > Sanity check passed.\n")

    # 2 - Init Model
    model = model_class(**model_args).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.CrossEntropyLoss()

    best_val_acc = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "val_top5": []
    }

    # 3 - TRAIN LOOP
    for epoch in range(epochs):
        print(f"\n{'='*60}")
        print(f" -- Epoch {epoch+1}/{epochs}")
        print(f"{'='*60}")

        # ---- TRAIN ----
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for x, y in train_loader:
            if x is None: continue

            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)

            loss = criterion(out, y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            total_loss += loss.item()

            preds = out.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = correct / total

        # ---- VALIDATION ----
        val_metrics = evaluate_model(
            model, val_loader, device,
            criterion, num_classes, epoch+1, "Val"
        )

        val_acc = val_metrics["accuracy"]
        val_loss = val_metrics["loss"]

        # ---- SAVE HISTORY ----
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_top5"].append(val_metrics["top5_accuracy"])

        print(f"\n📊 Epoch Summary:")
        print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc*100:.2f}%")
        print(f"Val   Loss: {val_loss:.4f}, Acc: {val_acc*100:.2f}%")

        scheduler.step()

        # ---- SAVE BEST ----
        if val_acc > best_val_acc:
            best_val_acc = val_acc

            torch.save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "val_acc": val_acc
            }, BEST_MODEL_PATH)

            print("✅ Saved new best model!")

            # Save confusion matrix
            plot_confusion_matrix(
                val_metrics["confusion_matrix"],
                epoch+1,
                "Val",
                os.path.join(save_dir, f"cm_epoch_{epoch+1}.png")
            )

    # -------------------------
    # 🔹 4. TRAINING DONE
    # -------------------------
    print("\n🏁 Training complete.")
    plot_metrics_history(history, os.path.join(save_dir, "training_curves.png"))

    # -------------------------
    # 🔹 5. LOAD BEST MODEL
    # -------------------------
    checkpoint = torch.load(BEST_MODEL_PATH)
    model.load_state_dict(checkpoint["model_state_dict"])

    # -------------------------
    # 🔹 6. TEST EVALUATION
    # -------------------------
    print("\n🧪 Running FINAL TEST evaluation...")

    test_metrics = evaluate_model(
        model, test_loader, device,
        criterion, num_classes, epoch="FINAL", phase="Test"
    )

    print("\n🎯 FINAL TEST RESULTS:")
    print(f"Top-1 Accuracy: {test_metrics['accuracy']*100:.2f}%")
    print(f"Top-5 Accuracy: {test_metrics['top5_accuracy']*100:.2f}%")
    print(f"F1 Macro: {test_metrics['f1_macro']*100:.2f}%")

    # Save results
    with open(FINAL_RESULTS_PATH, "w") as f:
        json.dump({
            "top1_accuracy": test_metrics["accuracy"],
            "top5_accuracy": test_metrics["top5_accuracy"],
            "f1_macro": test_metrics["f1_macro"]
        }, f, indent=4)

    print(f"\n💾 Results saved at: {FINAL_RESULTS_PATH}")

    return model, history, test_metrics